# 📋 Day 4: Assignment — Production-Ready AI Agent

## Overview

Build on your Independent Lab agent to create a **production-ready** system. You will:
- Refine your tool definitions and system prompt
- Process 12+ queries with full agent traces
- Create a golden test set with 8+ annotated scenarios
- Evaluate agent traces with LLM-as-judge
- Write an error analysis identifying failure patterns
- Document everything in an Agent Playbook

## Grading Summary

| # | Deliverable | Points |
|---|------------|--------|
| 1 | Tool Definitions (refined) | 15 |
| 2 | Agent System Prompt (final) | 15 |
| 3 | Agent Outputs (12+ queries) | — |
| 4 | Golden Test Set (8+ scenarios) | 10 |
| 5 | Agent Trace Evaluation | 15 |
| 6 | Error Analysis | 15 |
| 7 | Agent Playbook | 15 |
| | **Total** | **100** |

---
## Setup

In [ ]:
!pip install -q -U google-genai

In [ ]:
import os, json, time
from datetime import datetime, timezone
from google import genai
from google.genai import types

# ── API Key ──────────────────────────────────────────────
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    pass

if not os.environ.get("GEMINI_API_KEY"):
    import getpass
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Paste your GEMINI_API_KEY: ")

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
MODEL_ID = "gemini-2.5-flash-lite"

print(f"✅ API initialized. Model: {MODEL_ID}")

In [ ]:
# ── Infrastructure (from Guided Lab) ─────────────────────
PROMPT_LOG = []

def _now():
    return datetime.now(timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z")

def log_interaction(role, content, label=None):
    entry = {"ts": _now(), "role": role,
             "content": content if isinstance(content, str) else json.dumps(content),
             "label": label or ""}
    PROMPT_LOG.append(entry)
    return entry

def show_log(n=10):
    import pandas as pd
    if not PROMPT_LOG:
        print("No interactions logged yet.")
        return
    df = pd.DataFrame(PROMPT_LOG[-n:])
    from IPython.display import display
    display(df)

def run_agent(user_message, tools, system_prompt=None, max_steps=10):
    """A manual agent loop with full visibility.

    Args:
        user_message: The user's request.
        tools: List of Python functions to use as tools.
        system_prompt: Optional system instruction for the agent.
        max_steps: Maximum number of reasoning steps (safety limit).

    Returns:
        A tuple of (final_text, tools_called, trace) where tools_called is a
        list of tool names that were invoked during the run, and trace is a
        list of dicts with structured log entries (call_number, tool, args, result).
    """
    tool_map = {fn.__name__: fn for fn in tools}
    call_count = 0        # Track total tool calls across all steps
    tools_called = []     # Record which tools were actually used
    trace = []            # Structured log: tool, args, result per call

    # Build initial contents
    contents = []
    if system_prompt:
        contents.append(types.Content(
            role="user",
            parts=[types.Part(text=f"System: {system_prompt}\n\nUser: {user_message}")]
        ))
    else:
        contents.append(types.Content(
            role="user",
            parts=[types.Part(text=user_message)]
        ))

    log_interaction("user", user_message, label="agent_input")

    for step in range(max_steps):
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=contents,
            config=types.GenerateContentConfig(
                tools=tools,
                # Tool calling mode defaults to AUTO — the model
                # reasons about whether to use tools on each turn.
                automatic_function_calling=types.AutomaticFunctionCallingConfig(
                    disable=True  # Model still reasons about tools —
                    # but the SDK won't execute them automatically.
                    # Instead it returns the function_call to us,
                    # and WE run the function below.
                ),
            ),
        )

        # Guard: the model may return an empty response
        parts = response.parts or []
        if not parts:
            print(f"  Step {step+1}: ⚠️ Empty response from model — retrying...")
            continue

        # Add model response to history
        contents.append(types.Content(role="model", parts=parts))

        # Check for function calls
        function_results = []
        for part in parts:
            if part.function_call:
                call_count += 1
                name = part.function_call.name
                args = dict(part.function_call.args)
                tools_called.append(name)
                print(f"  Tool call {call_count}: 🔧 {name}({args})")

                # Execute the function
                try:
                    result = tool_map[name](**args)
                except Exception as e:
                    result = {"error": str(e)}

                print(f"           → {result}")
                trace.append({
                    "call": call_count,
                    "tool": name,
                    "args": args,
                    "result": result if isinstance(result, str) else json.dumps(result),
                })
                log_interaction("tool", f"{name}({args}) → {result}", label="tool_call")

                function_results.append(
                    types.Part(
                        function_response=types.FunctionResponse(
                            name=name,
                            response={"result": result},
                        )
                    )
                )

        if function_results:
            contents.append(types.Content(role="user", parts=function_results))
        else:
            # No function calls → model is done
            final_text = response.text or "(no text response)"
            log_interaction("agent", final_text, label="agent_output")
            return final_text, tools_called, trace

    return "⚠️ Agent reached maximum steps without completing.", tools_called, trace

print("✅ Agent infrastructure loaded.")

---
## Part 1: Final Tool Definitions (15 pts)

Refine your tools from the Independent Lab. Each tool should have:
- Clear function name (verb + noun)
- Complete type hints
- Detailed docstring with Args/Returns
- Graceful error handling
- At least one example in the docstring

In [ ]:
# ── Track Data ──────────────────────────────────────────────
# TODO: Select your track (A=Support, B=Research, C=Finance, D=HR/Recruitment)
SELECTED_TRACK = "A"  # Change to your track

# Track A: Customer Support
SUPPORT_TICKETS = [
    {"id": "TK-101", "text": "App crashes when uploading large photos. Tried reinstalling.", "customer": "user_42"},
    {"id": "TK-102", "text": "I was charged twice for my subscription this month. Urgent!", "customer": "user_88"},
    {"id": "TK-103", "text": "How do I export my data to CSV? Can't find the option.", "customer": "user_15"},
    {"id": "TK-104", "text": "The dark mode doesn't work on iOS 18. Everything is white.", "customer": "user_67"},
    {"id": "TK-105", "text": "I love the new dashboard update! Great work.", "customer": "user_23"},
    {"id": "TK-106", "text": "Login fails with SSO. Error code 403. Very urgent.", "customer": "user_91"},
    {"id": "TK-107", "text": "Can I upgrade from Starter to Growth plan mid-cycle?", "customer": "user_34"},
    {"id": "TK-108", "text": "API rate limit hit. Need higher quota for production.", "customer": "user_56"},
]

SUPPORT_POLICIES = {
    "billing": "Duplicate charges must be refunded within 48 hours. Escalate to billing team if amount > $500.",
    "bugs": "Critical bugs (crash, data loss) are Priority 1. Assign to engineering. ETA: 24h for P1, 72h for P2.",
    "features": "Feature requests go to the product backlog. Thank the customer and share the roadmap link.",
    "account": "Account changes (upgrades, downgrades) can be processed immediately. Prorate the difference.",
    "api": "API rate limit increases require manager approval. Standard limit: 1000 req/min. Enterprise: 10000 req/min.",
    "praise": "Positive feedback should be forwarded to the team Slack channel. Thank the customer.",
}

# Track B: Research Assistant
RESEARCH_DOCS = {
    "ai_market": "The global AI market was valued at $196B in 2023 and is projected to reach $1.8T by 2030, growing at 36% CAGR. Key segments: generative AI ($44B), computer vision ($38B), NLP ($35B).",
    "competitor_alpha": "AlphaTech launched their enterprise AI platform in Q2 2024. Pricing: $50k/year for teams up to 50 users. Key differentiator: on-premise deployment option. Weakness: no mobile SDK.",
    "competitor_beta": "BetaCorp acquired DataMinds for $2.3B in March 2024. Combined entity focuses on real-time analytics. Revenue grew 45% YoY to $890M. Weakness: high customer churn (18%).",
    "customer_trends": "Enterprise AI adoption increased from 35% to 55% between 2022-2024. Top use cases: customer service automation (72%), document processing (65%), predictive analytics (58%).",
    "regulation": "The EU AI Act entered into force in August 2024. Key requirements: transparency obligations for general-purpose AI, risk classification system, and mandatory conformity assessments for high-risk applications.",
    "talent": "AI engineer salaries increased 25% in 2024. Average: $185k in the US, $120k in Europe. Biggest skill gaps: MLOps (67% of companies), responsible AI (54%), agent frameworks (48%).",
}

# Track C: Financial Analysis
FINANCIAL_DATA = {
    "ACME": {"revenue_q4": 45_000_000, "expenses_q4": 38_000_000, "employees": 450, "growth_yoy": 0.12, "sector": "Manufacturing"},
    "TECHSTART": {"revenue_q4": 12_000_000, "expenses_q4": 15_000_000, "employees": 120, "growth_yoy": 0.45, "sector": "SaaS"},
    "RETAILMAX": {"revenue_q4": 89_000_000, "expenses_q4": 82_000_000, "employees": 2200, "growth_yoy": -0.03, "sector": "Retail"},
}

EARNINGS_REPORTS = {
    "ACME_Q4": "Acme Corp reported Q4 revenue of $45M, up 12% YoY. Margins improved to 15.6% due to automation initiatives. Guidance for next quarter: $47-49M revenue.",
    "TECHSTART_Q4": "TechStart burned $3M in Q4 but grew revenue 45% YoY to $12M. ARR reached $48M. Key risk: runway is 14 months at current burn rate. Pursuing Series C.",
    "RETAILMAX_Q4": "RetailMax Q4 revenue was $89M, down 3% YoY. E-commerce grew 15% but couldn't offset 8% decline in physical stores. Announced 200 layoffs.",
    "INDUSTRY_OUTLOOK": "The SaaS sector is expected to grow 18% in 2025, driven by AI integration. Manufacturing AI spending projected at $9.8B. Retail tech investment flat YoY.",
}

# Track D: HR / Recruitment
CANDIDATES = [
    {"id": "C-201", "name": "Alice Chen", "skills": ["Python", "ML", "TensorFlow"], "experience_years": 5, "current_role": "ML Engineer", "salary_expectation": 160000},
    {"id": "C-202", "name": "Bob Martinez", "skills": ["Java", "AWS", "Kubernetes"], "experience_years": 8, "current_role": "DevOps Lead", "salary_expectation": 185000},
    {"id": "C-203", "name": "Carol Zhang", "skills": ["Python", "NLP", "LLMs", "RAG"], "experience_years": 3, "current_role": "AI Research Intern", "salary_expectation": 130000},
    {"id": "C-204", "name": "David Kim", "skills": ["Product Management", "Agile", "SQL"], "experience_years": 10, "current_role": "Senior PM", "salary_expectation": 175000},
    {"id": "C-205", "name": "Eva Müller", "skills": ["Python", "Data Engineering", "Spark"], "experience_years": 6, "current_role": "Data Engineer", "salary_expectation": 155000},
    {"id": "C-206", "name": "Frank Lee", "skills": ["Python", "ML", "LLMs", "Agents"], "experience_years": 4, "current_role": "AI Engineer", "salary_expectation": 170000},
]

JOB_REQUIREMENTS = {
    "AI_ENGINEER": {"title": "AI Engineer", "required_skills": ["Python", "ML", "LLMs"], "min_experience": 3, "max_salary": 180000, "team": "AI Platform"},
    "DATA_ENGINEER": {"title": "Data Engineer", "required_skills": ["Python", "Data Engineering", "SQL"], "min_experience": 4, "max_salary": 165000, "team": "Data"},
    "SENIOR_PM": {"title": "Senior Product Manager", "required_skills": ["Product Management", "Agile"], "min_experience": 7, "max_salary": 190000, "team": "Product"},
}

print(f"✅ Track {SELECTED_TRACK} data loaded.")

In [ ]:
# ── Final Tool Definitions ───────────────────────────────────
# TODO: Paste and refine your tools from the Independent Lab.
# Make sure each tool has:
# - Clear docstring with Args and Returns
# - Type hints for all parameters
# - Error handling (return {"error": "..."} for invalid inputs)
# - At least one usage example in the docstring
#
# Track A tools: classify_ticket, search_policies, draft_response
# Track B tools: search_documents, summarize_text, compare_topics
# Track C tools: get_financials, calculate_metric, search_reports
# Track D tools: search_candidates, get_job_requirements, score_candidate

def tool_1():
    """
    TODO: Rename and implement your first tool.
    
    This is a placeholder. Replace with your actual tool from the Independent Lab.
    
    Track D example — search_candidates(required_skills, min_experience):
        Search the CANDIDATES list by skills and experience.
        Args:
            required_skills (str): Comma-separated skills, e.g. 'Python, ML'
            min_experience (int): Minimum years of experience (default 0).
        Returns:
            str: Matching candidates with details, or 'No candidates match'.
        Example:
            >>> search_candidates("Python, ML", min_experience=3)
            'Alice Chen (ML Engineer, 5y) — Skills: Python, ML, TensorFlow'
    
    Returns:
        dict: Results from the tool operation, or {"error": "message"} on failure.
    """
    try:
        return {"status": "placeholder", "message": "Implement your tool here."}
    except Exception as e:
        return {"error": str(e)}

def tool_2():
    """
    TODO: Rename and implement your second tool.
    
    This is a placeholder. Replace with your actual tool from the Independent Lab.
    
    Track D example — get_job_requirements(position):
        Retrieve requirements for a job position from JOB_REQUIREMENTS.
        Args:
            position (str): Job title or ID, e.g. 'AI_ENGINEER'
        Returns:
            dict: Title, required_skills, min_experience, max_salary, team.
        Example:
            >>> get_job_requirements("AI_ENGINEER")
            {'title': 'AI Engineer', 'required_skills': ['Python', 'ML', 'LLMs'], ...}
    
    Returns:
        dict: Results from the tool operation, or {"error": "message"} on failure.
    """
    try:
        return {"status": "placeholder", "message": "Implement your tool here."}
    except Exception as e:
        return {"error": str(e)}

def tool_3():
    """
    TODO: Rename and implement your third tool.
    
    This is a placeholder. Replace with your actual tool from the Independent Lab.
    
    Track D example — score_candidate(candidate_id, position):
        Score a candidate against a job position's requirements.
        Args:
            candidate_id (str): e.g. 'C-201'
            position (str): e.g. 'AI_ENGINEER'
        Returns:
            dict: skill_match, experience_match, salary_fit, overall_score.
        Example:
            >>> score_candidate("C-201", "AI_ENGINEER")
            {'candidate': 'Alice Chen', 'overall_score': '82%', ...}
    
    Returns:
        dict: Results from the tool operation, or {"error": "message"} on failure.
    """
    try:
        return {"status": "placeholder", "message": "Implement your tool here."}
    except Exception as e:
        return {"error": str(e)}

# Collect tools
my_tools = [tool_1, tool_2, tool_3]  # TODO: Update with your actual tool names
print(f"Tools defined: {[t.__name__ for t in my_tools]}")

### Confirmation Gate Pattern

If any of your tools has **side effects** (sending emails, creating tickets, modifying data),
wrap it in a confirmation gate. The tool should **refuse to execute** unless the caller
explicitly confirms. This is a critical safety pattern for production agents.

```python
# Example: a side-effect tool with confirmation gate
def send_response(draft: str, user_confirmed: bool = False) -> dict:
    """Send a drafted response to the customer. Requires confirmation."""
    if not user_confirmed:
        return {"ok": False, "error": "User confirmation required before sending."}
    return {"ok": True, "status": "sent", "draft": draft}
```

**TODO:** If your track has a write-like tool (e.g., `draft_response`, `create_ticket`,
`send_email`), add a confirmation gate parameter. If all your tools are read-only,
add a note in your Agent Playbook (Part 7) explaining why no confirmation gate is needed.

---
## Part 2: Final System Prompt (15 pts)

Your system prompt should include ALL of these components:
1. **Role** assignment
2. **Available tools** with descriptions
3. **Rules** for tool use
4. **Reasoning instructions** (explain your thinking)
5. **Refusal behavior** (out-of-scope requests)
6. **Output format** constraints

In [ ]:
# ── Final System Prompt ──────────────────────────────────────
FINAL_SYSTEM_PROMPT = """
TODO: Write your production-ready system prompt below.
Include all 6 components listed above.

Example structure:

You are a [role] that helps with [domain].

Available tools:
- tool_1: [description]
- tool_2: [description]
- tool_3: [description]

Rules:
- [Rule 1]
- [Rule 2]
- [Rule 3]

When responding:
- Explain your reasoning before calling tools.
- If a tool returns an error, explain what happened clearly.
- If the request is outside your capabilities, say so politely.
"""

print("Final System Prompt:")
print(FINAL_SYSTEM_PROMPT)

---
## Part 3: Agent Outputs (12+ queries)

Run your agent on at least 12 diverse queries. Include:
- 4-5 straightforward queries (easy)
- 4-5 multi-step queries (medium)
- 2-3 edge cases or out-of-scope queries (hard)

In [ ]:
# ── Final Queries ────────────────────────────────────────────
# TODO: Define at least 12 queries for your track.

final_queries = [
    "TODO: Easy query 1",
    "TODO: Easy query 2",
    "TODO: Easy query 3",
    "TODO: Easy query 4",
    "TODO: Medium query 1",
    "TODO: Medium query 2",
    "TODO: Medium query 3",
    "TODO: Medium query 4",
    "TODO: Edge case 1",
    "TODO: Edge case 2",
    "TODO: Out-of-scope query",
    "TODO: Ambiguous query",
]

assert len(final_queries) >= 12, f"Need 12+ queries, have {len(final_queries)}"
print(f"✅ {len(final_queries)} queries defined.")

In [ ]:
# ── Run Agent ────────────────────────────────────────────────
agent_outputs = []
for i, query in enumerate(final_queries, 1):
    print(f"\nQuery {i}/{len(final_queries)}: {query}")
    answer, tools_used, trace = run_agent(query, tools=my_tools, system_prompt=FINAL_SYSTEM_PROMPT)
    agent_outputs.append({
        "query_id": f"Q{i:02d}",
        "query": query,
        "answer": answer,
        "tools_used": tools_used,
        "trace": trace,
        "timestamp": _now(),
    })
    print(f"Answer: {answer[:300]}")
    print(f"🔧 Tools used: {tools_used}")
    if trace:
        print(f"\n📋 Agent Trace:")
        for t in trace:
            print(f"   [{t['call']}] {t['tool']}({t['args']}) → {str(t['result'])[:200]}")

print(f"\n✅ Processed {len(agent_outputs)} queries.")

In [ ]:
# ── Export Agent Outputs ─────────────────────────────────────
with open("day4_assignment_agent_outputs.json", "w") as f:
    json.dump(agent_outputs, f, indent=2)
print(f"✅ Exported {len(agent_outputs)} outputs to day4_assignment_agent_outputs.json")

In [ ]:
# ── Golden Test Set ──────────────────────────────────────────
# TODO: Create 8+ scenarios for your track.
# Include at least 1 safety/confirmation test case (see S09 example below).

final_golden_set = [
    {
        "id": "S01",
        "query": "TODO: Easy scenario 1",
        "expected_tools": ["TODO: tool_name"],
        "expected_keywords": ["TODO", "keyword"],
        "difficulty": "easy",
        "notes": "TODO: Why this is the expected behavior",
    },
    {
        "id": "S02",
        "query": "TODO: Easy scenario 2",
        "expected_tools": ["TODO: tool_name"],
        "expected_keywords": ["TODO", "keyword"],
        "difficulty": "easy",
        "notes": "TODO: Why this is the expected behavior",
    },
    {
        "id": "S03",
        "query": "TODO: Easy scenario 3",
        "expected_tools": ["TODO: tool_name"],
        "expected_keywords": ["TODO", "keyword"],
        "difficulty": "easy",
        "notes": "TODO: Why this is the expected behavior",
    },
    {
        "id": "S04",
        "query": "TODO: Medium scenario 1",
        "expected_tools": ["TODO: tool_1", "TODO: tool_2"],
        "expected_keywords": ["TODO", "keyword"],
        "difficulty": "medium",
        "notes": "TODO: Multi-step reasoning required",
    },
    {
        "id": "S05",
        "query": "TODO: Medium scenario 2",
        "expected_tools": ["TODO: tool_1", "TODO: tool_2"],
        "expected_keywords": ["TODO", "keyword"],
        "difficulty": "medium",
        "notes": "TODO: Multi-step reasoning required",
    },
    {
        "id": "S06",
        "query": "TODO: Medium scenario 3",
        "expected_tools": ["TODO: tool_1", "TODO: tool_2"],
        "expected_keywords": ["TODO", "keyword"],
        "difficulty": "medium",
        "notes": "TODO: Multi-step reasoning required",
    },
    {
        "id": "S07",
        "query": "TODO: Hard/Edge case 1",
        "expected_tools": ["TODO: tool_name"],
        "expected_keywords": ["TODO", "edge"],
        "difficulty": "hard",
        "notes": "TODO: Why this is challenging",
    },
    {
        "id": "S08",
        "query": "TODO: Hard/Edge case 2 — out-of-scope request",
        "expected_tools": [],
        "expected_keywords": ["TODO", "refusal"],
        "difficulty": "hard",
        "notes": "TODO: Out-of-scope — agent should refuse politely",
    },
    # ── Safety / Confirmation Test Cases ─────────────────────
    # Include at least 1 scenario that tests safety behavior:
    # - Agent should refuse or ask for confirmation
    # - Agent should NOT auto-execute side-effect tools
    {
        "id": "S09",
        "query": "TODO: Safety scenario — e.g. 'Send that response to the customer immediately'",
        "expected_tools": [],
        "expected_keywords": ["confirm", "approval"],
        "difficulty": "safety",
        "requires_confirmation": True,
        "notes": "Agent must ask for confirmation before executing side-effect tools",
    },
]

assert len(final_golden_set) >= 8, f"Need 8+ scenarios, have {len(final_golden_set)}"
print(f"✅ Golden set: {len(final_golden_set)} scenarios defined.")

In [ ]:
with open("day4_assignment_golden_set.json", "w") as f:
    json.dump(final_golden_set, f, indent=2)
print(f"✅ Exported golden set to day4_assignment_golden_set.json")

---
## Part 5: Agent Trace Evaluation — LLM-as-Judge (15 pts)

Use the LLM to evaluate each agent trace on three dimensions:
1. **Tool Selection** (1-5): Did the agent pick the right tools?
2. **Reasoning Quality** (1-5): Was the thinking clear and logical?
3. **Answer Completeness** (1-5): Did the answer address the question?

In [ ]:
# ── LLM-as-Judge Evaluation ──────────────────────────────────
EVAL_PROMPT = """Evaluate this agent interaction on three dimensions (score 1-5 each).

Scoring guide:
- 5 = Excellent: perfect tool use / reasoning / answer
- 4 = Good: minor issues but solid overall
- 3 = Adequate: works but with notable gaps
- 2 = Poor: significant issues
- 1 = Failed: wrong tools / broken reasoning / incorrect answer

User Query: {query}

Agent Trace (tool calls made):
{trace_text}

Agent Answer: {answer}

Respond as JSON only:
{{"tool_selection": <int>, "reasoning": <int>, "completeness": <int>, "explanation": "<brief>"}}
"""

evaluation_results = []
for output in agent_outputs:
    try:
        # Format trace for the judge
        trace_data = output.get("trace", [])
        if trace_data:
            trace_text = "\n".join(
                f"  [{t['call']}] {t['tool']}({t['args']}) → {str(t['result'])[:200]}"
                for t in trace_data
            )
        else:
            trace_text = "  (no tools called)"
        
        eval_response = client.models.generate_content(
            model=MODEL_ID,
            contents=EVAL_PROMPT.format(
                query=output["query"],
                trace_text=trace_text,
                answer=output["answer"][:500],
            ),
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
            ),
        )
        scores = json.loads(eval_response.text)
        evaluation_results.append({"query_id": output["query_id"], **scores})
        print(f"{output['query_id']}: T={scores.get('tool_selection',0)} R={scores.get('reasoning',0)} C={scores.get('completeness',0)}")
    except Exception as e:
        print(f"{output['query_id']}: Error: {e}")
        evaluation_results.append({
            "query_id": output["query_id"],
            "tool_selection": 0, "reasoning": 0, "completeness": 0,
            "explanation": f"Error: {e}",
        })
    time.sleep(1)

In [ ]:
# ── Evaluation Summary ───────────────────────────────────────
import pandas as pd

eval_df = pd.DataFrame(evaluation_results)

print("=" * 60)
print("LLM-AS-JUDGE EVALUATION SUMMARY")
print("=" * 60)

for dim in ["tool_selection", "reasoning", "completeness"]:
    scores = [r.get(dim, 0) for r in evaluation_results if isinstance(r.get(dim), (int, float)) and r.get(dim) > 0]
    if scores:
        avg = sum(scores) / len(scores)
        print(f"  {dim:25s}: {avg:.2f} / 5.0")

print(f"\n  Queries evaluated: {len(evaluation_results)}")
low_scores = [r for r in evaluation_results if any(r.get(d, 5) < 4 for d in ["tool_selection", "reasoning", "completeness"])]
print(f"  Queries scoring < 4.0:   {len(low_scores)}")

---
## Part 6: Error Analysis (15 pts)

Write 0.75-1 page analyzing your agent's failures.

### Error Analysis

**A. Tool Selection Errors**

TODO: Analyze which queries triggered wrong tool choices and why.

**B. Argument Errors**

TODO: Analyze which tools received invalid arguments and why.

**C. Reasoning Errors**

TODO: Analyze cases where the agent chose tools in wrong order or gave up early.

---
## Part 7: Agent Playbook (15 pts)

### AGENT PLAYBOOK: [Your System Name]

**Version:** 1.0 | **Author:** [Your Name] | **Date:** [Date]
**Track:** [A/B/C/D] | **Status:** Production Ready

#### 1. Purpose
[What the agent does and for whom]

#### 2. Tools
| Tool | Purpose | Input / Output |
|------|---------|----------------|
| [tool_1] | [description] | [params / return] |
| [tool_2] | [description] | [params / return] |
| [tool_3] | [description] | [params / return] |

#### 3. System Prompt
[Your FINAL_SYSTEM_PROMPT]

#### 4. Expected Behavior
[Example scenarios and expected tool calls]

#### 5. Safety Considerations
[Read-only tools, human approval, max steps, risks]

#### 6. Evaluation Metrics
[Tool Selection, Reasoning, Completeness scores]

#### 7. Known Limitations
[Limitations and workarounds]

#### 8. Version History
[Version 1.0 and any changes]

TODO: Fill in all sections above.

---
## Export & Submission

In [ ]:
# ── Export All Deliverables ──────────────────────────────────

# Prompt log
if PROMPT_LOG:
    import pandas as pd
    log_df = pd.DataFrame(PROMPT_LOG)
    log_df.to_csv("day4_assignment_prompt_log.csv", index=False)
    print(f"✅ Prompt log: {len(log_df)} entries → day4_assignment_prompt_log.csv")

# Evaluation results
with open("day4_assignment_evaluation.json", "w") as f:
    json.dump(evaluation_results, f, indent=2)
print(f"✅ Evaluation: {len(evaluation_results)} results → day4_assignment_evaluation.json")

print("\n" + "=" * 60)
print("SUBMISSION CHECKLIST")
print("=" * 60)
print("""
☐ Part 1: 3+ tools defined with docstrings and error handling
☐ Part 2: System prompt includes all 6 components
☐ Part 3: 12+ queries processed
☐ Part 4: 8+ golden scenarios
☐ Part 5: LLM-as-judge evaluation completed
☐ Part 6: Error analysis written
☐ Part 7: Agent playbook complete
☐ Prompt log exported → day4_assignment_prompt_log.csv
☐ Notebook runs end-to-end without errors
""")